### Data Preparation


**1. Missing data and missingness mechanisms**

1. MCAR - Deletion (if small %), mean/median/mode imputation (simple, biases variance), model-based imputation (KNN/MICE, more accurate), or flag-as-category (tree models can use missingness itself as signal).

2. MAR: depends on observed variables — impute using them (MICE, model-based). ex- Example: Older patients are less likely to report their income, but "being older" is something you can observe (age column exists). Missingness in income depends on age (observed), not on income itself.

3. MNAR — missingness depends on the missing value itself, so mean/median imputation biases results; model the mechanism or run sensitivity analysis and add an indicator. ex - Example: People with very high income don't report it (privacy). The missingness in income depends on income itself. You can't see this pattern just by looking at other columns.

Case 1: Too much missing data (>40-50%) - If a column is mostly missing, imputing fills in mostly "guessed" values. Better to drop the column entirely, or treat missingness as a feature.

Case 2: MNAR with no reliable domain signal - If you don't know WHY data is missing and can't model it, imputing might just bake in your assumptions as "fact." Better to be transparent: use missingness indicators, don't pretend you know the true value.

Case 3: Target leakage risk - If missingness is caused by the OUTCOME you're trying to predict
(e.g., "churned customers stopped filling the survey") . Imputing based on that survey might leak future information.


Scenario : Model to predict credit default but 20% of the values are missing (NaN). A Naive Median Imputation (replacing all missing values with the median credit score of $680$) AND add a binary column is_missing ($1$ if missing, $0$ otherwise). $y = \sigma(w_1 \cdot \text{credit\_score} + w_2 \cdot \text{is\_missing} + b)$. Model learns a dedicated coefficient $w_2$ specifically for the missing group! This $w_2$ acts as an offset that absorbs the bias introduced by substituting the constant,

Scneario : Building a credit default model. You notice that 40% of applicants have NaN for income_verification_document .Group A (Missing): Default rate is 18%. Group B (Provided): Default rate is 4%. If an upstream product team updates the mobile app user interface so that income_verification_document becomes optional for high-income pre-approved users, how will this change affect your model's predictions, and why? 
    - Model will overestimate Risk of deafult, Leading to high FP. As during Training, MNAR, tree exploited this feature as missingness was correlated to high risk behaviour. Create an explicit indicator feature like is_document_waived_by_policy rather than passing a raw NaN

In [ ]:
import pandas as pd
from scipy import stats

df = pd.DataFrame({
    'age': [25, 30, 22, 35, 28, 40, 29, 31],
    'income': [50000, 60000, None, 80000, 55000, None, 70000, 65000]
})

df['income_missing'] = df['income'].isna().astype(int)
df.groupby('income_missing')['age'].mean()

# Statistical Test - Little's MCAR Test
# Test NUll Hypothesis: The mean age of individuals with missing income data is equal to the mean age of individuals with non-missing income data.
# if p-value < 0.05, we reject the null hypothesis and conclude that there is a significant difference in mean age between the two groups.

# difference in age between the two groups → missingness is NOT random w.r.t. age → suggests MAR (not MCAR)

group_missing = df[df['income_missing'] == 1]['age']
group_present = df[df['income_missing'] == 0]['age']

t_stat, p_value = stats.ttest_ind(group_missing, group_present, equal_var=False)
# Welch's t-test, doesn't assume the two groups have equal variance. Safer default than the standard t-test when group sizes/variances are small or unequal (as here, with only 2 missing rows).

print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.3f}")

if p_value < 0.05:
    print("Reject null hypothesis: significant age difference between groups → likely MAR")
else:
    print("Fail to reject null hypothesis: no significant age difference → consistent with MCAR")

In [ ]:
# MAR: Use Other Variables to Predict

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# MICE (Multiple Imputation by Chained Equations)
imputer = IterativeImputer(random_state=42)
df_imputed = imputer.fit_transform(df)

# Why this works: MAR means missingness depends on OTHER columns,
# so using those columns to predict the missing value removes the bias

# -----------------------------#

# MNAR: Tricky — Standard Imputation Introduces Bias
# a) Add a "missingness indicator" as a feature
# Let the model learn: "missingness itself is informative"
# (e.g., tree-based models can split on this flag)

df['income_was_missing'] = df['income'].isna().astype(int)
df['income'] = df['income'].fillna(df['income'].median())

# b. If you know WHY data is missing (e.g., "privacy opt-out for income > $200k"),
# impute with a value informed by that domain knowledge
# (e.g., impute at the 90th percentile, not the mean)

# c) Try multiple imputation strategies (optimistic, pessimistic, mean)
# and see how much your conclusions change

# d) Don't impute — flag as a separate category (for categorical data)

**2. Outliers**

1. IQR rule - (below Q1-1.5×IQR or above Q3+1.5×IQR). Q1- 25th Percentile , Q3 - 75th Percentile.
    ```
    # Calculate Q1, Q3, and IQR
        q1 = df_capped[column].quantile(0.25)
        q3 = df_capped[column].quantile(0.75)
        iqr = q3 - q1
    ```
2. Z-score - $\Vert{}Z\Vert{} > 3$ where $Z = \frac{x - \mu}{\sigma}$. Best Used for Normally distrubited data. Measue How many SD a data point is away from mean. Why 3 ? - 68-95-99.7 Rule. 99.7% of data lies with +- 3 S.D.

3. Isolation Forest - When high dimensional data, Tree-based isolation score. Unlike single-feature IQR or Z-score, Isolation Forest isolates anomalies across joint feature spaces (e.g., detecting points where Salary is abnormal relative to Age).
    ```
    # 2. Fit Isolation Forest Model
    # contamination: estimated fraction of outliers in dataset
    iso_forest = IsolationForest(contamination=0.02, random_state=42)
    predictions = iso_forest.fit_predict(df[["Age", "Salary"]])

    # Isolation Forest returns: 1 for inliers, -1 for anomalies
    df["Anomaly_Label"] = np.where(predictions == -1, "Outlier", "Inlier")
    ```

4. DBSCAN - Spatial or Cluster Data. Points categorized as noise

*Techniques :*

- Trimming or Removal, Capping or Winsorizing (Replacing Values above 99th percentile or below 1st percentile - It preserves the data size), Transformatio (log(X) or Box-Cox to shrink large values & normalize skewed distributions) `df_capped[column] = np.clip(df_capped[column], lower_bound, upper_bound)`

- Imputation Techniques, Treating missing values as NaN

- Use of Robust Algorithms - RF, XGBoost, Robust Regression (Huber Loss)


**3. Feature scaling**

1. Standardization (x−μ)/σ: zero mean/unit variance — default for linear models, SVM, NN, PCA. Especially for those algorithm that assumes normally distributed data. Does not bound data to a fixed range

2. Min-max: [0,1], outlier-sensitive (images/NN/KNN). $x' = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$ . Distance-based algorithms when data does not follow a Gaussian distribution—$k$-Nearest Neighbors ($k$-NN), $k$-Means Clustering, and image processing (pixel values $0–255$).

3. Normalization (L1 / L2 Vector Normalization)- Scales individual rows (samples) so that each sample vector has a unit norm (length of 1). Best Used For: Text classification, document clustering (TF-IDF vectors), and sparse datasets where row-wise direction matters more than feature magnitude. Formulas:
    - $L1$ Norm (Manhattan): $x' = \frac{x}{\sum \vert{}x_i\vert{}}$ 
    - $L2$ Norm (Euclidean): $x' = \frac{x}{\sqrt{\sum x_i^2}}$


**4. Feature Engineering**

1. One-Hot Encoding (OHE) - Creates a binary dummy variable ($0$ or $1$) for each unique category level. Nominal data with low cardinality (fewer than $10–15$ unique categories). Dummy Variable Trap: Creates $K$ columns for $K$ categories. Drop one column ($K-1$ encoded columns) when using linear models to prevent multi-collinearity.

2. Ordinal Encoding - Maps each ordered category to an integer based on its explicit rank. Ordinal features where hierarchy carries meaningful predictive signal.

3. Label Encoding - Assigns arbitrary sequential integers ($0, 1, 2, \dots, N-1$) to categorical labels. Best Used For: Target variables ($y$) in classification tasks (LabelEncoder in Scikit-Learn).

4. Target / Mean Encoding - Replaces each category with the mean value of the target variable ($y$) for that category.High-cardinality nominal features (e.g., ZipCode, Brand).High risk of data leakage and overfitting; requires smoothing or out-of-fold calculation.

5. Frequency / Count Encoding - Replaces each category with its total count or relative frequency in the dataset.	High-cardinality nominal features where category commonality matters.	Collapses distinct categories into the same code if they have identical frequencies.

In [ ]:
import pandas as pd

df = pd.DataFrame({"contact_channel": ["dating app", "email", "dating app", "social media DM"]})

print("one-hot:\n", pd.get_dummies(df["contact_channel"]))
print("\nlabel encoding:\n", df["contact_channel"].astype("category").cat.codes)
print("\nfrequency encoding:\n", df["contact_channel"].map(df["contact_channel"].value_counts()))

**5. Text Features**

1. Stemming: Uses crude heuristic rules to chop off word ends (e.g., "running", "runs" $\rightarrow$ "run"; "studies" $\rightarrow$ "studi"). Faster, but can produce non-dictionary words.

2. Lemmatization: Uses vocabulary and morphological analysis to return the true dictionary root (e.g., "better" $\rightarrow$ "good"; "studies" $\rightarrow$ "study").

3. N-grams: Preserves local word order by grouping adjacent words: Unigram: "great movie" $\rightarrow$ ["great", "movie"] Bigram: "great movie" $\rightarrow$ ["great movie"]. Limitation: Creates large, sparse matrices and ignores word meaning or context

4. Bag-of-Words (CountVectorizer) - Represents text by counting the frequency of each word across the vocabulary.

5. TF-IDF (Term Frequency - Inverse Document Frequency)- Scales word counts by how unique a word is across all documents: Strength: Automatically downweights ubiquitous words (like "movie" in movie reviews) while boosting unique key terms. $$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$ $$\text{IDF}(t) = \log\left(\frac{N}{\text{DF}(t)}\right)$$
    - TF (Term Frequency): Frequency of word $t$ in document $d$.  `tf(t, d) = count(t in d) / total_words(d)`
    - IDF (Inverse Document Frequency): Penalizes words that appear everywhere across $N$ documents ($\text{DF}(t)$). N = total documents in the corpus, df(t) = number of documents containing term t at least once (not how many times, just whether it appears at all).

> TF-IDF and bag-of-words treat every word independently, losing word order entirely

> **Why the log**, — without it, IDF would be a raw ratio `N/df(t)`. For a word appearing in just 1 of 1000 documents, that ratio is 1000, while a word in 500 of 1000 is only 2 — the rare word would dominate the score by a factor of 500, overweighting extremely rare (sometimes just noisy or misspelled) terms. The log compresses that: log(1000)≈6.9 vs log(2)≈0.7, a ratio of ~10x instead of 500x — rare terms still get more weight

6. A longer document naturally accumulates a bigger raw TF-IDF vector (more nonzero entries, more total "mass")  L2-normalizing divides each document's vector by its own magnitude, so every document vector has length 1 — comparisons become about the *direction* (which words matter, in what proportion) rather than raw magnitude.

7. Vocabulary is fixed at fit time — a word never seen during `fit` simply contributes nothing at inference. Interpretability (every dimension is a  word), no training required beyond counting, and strong performance  when the task is close to keyword matching

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = ["not a scam", "definitely a scam"]
vectorizer = CountVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(docs)
print("vocabulary:", vectorizer.get_feature_names_out())
print(X.toarray())

import math

def document_frequency(word, all_docs):
    return sum(1 for doc in all_docs if word in doc)

def idf(word, all_docs):
    N = len(all_docs)
    df = document_frequency(word, all_docs)
    return math.log(N / df)

for word in ["the", "cat", "sat", "on", "mat"]:
    print(f"idf({word}) = {idf(word, docs):.3f}")

In [ ]:
import numpy as np

def l2_normalize(vector_dict):
    values = np.array(list(vector_dict.values()))
    norm = np.sqrt(np.sum(values ** 2))
    return {k: v / norm for k, v in vector_dict.items()}

doc1_normalized = l2_normalize(doc1_vector)
for word, score in sorted(doc1_normalized.items(), key=lambda x: -x[1]):
    print(f"{word}: {score:.3f}")

# sanity check: normalized vector's own L2 norm should be 1.0
check = np.sqrt(sum(v**2 for v in doc1_normalized.values()))
print("\nnorm of normalized vector:", check)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(), index=["doc1", "doc2", "doc3"])
print(df.round(3))

print("\ndoc1, sorted by score (compare ordering to our from-scratch result):")
print(df.loc["doc1"].sort_values(ascending=False).head(5))

**6. Modern Static Embeddings**

**Word2Vec**

1. Word2Vec learns word meanings by scanning text and observing which words frequently appear next to each other. Word2Vec uses a simple 2-layer neural network trained on one of two tasks. 
    - CBOW (Continuous Bag of Words):Goal: Predicts the target word given its surrounding context words.Example: Inputs ["The", "cat", "___", "on", "mat"] $\rightarrow$ Predicts "sat".Best For: Faster to train; works well for frequent, common words.
    - Skip-Gram:Goal: Predicts the surrounding context words given a single target word.Example: Input "sat" $\rightarrow$ Predicts ["The", "cat", "on", "mat"].Best For: Performs better on smaller datasets and handles rare words well

2. Lightweight, extremely fast to train/infer, and captures semantic relationships far better than simple word counts or TF-IDF. Polysemy Problem: Every word gets only one fixed vector regardless of context (e.g., "bank" in river bank vs. savings bank uses the exact same numbers). Out-of-Vocabulary (OOV): Cannot handle unseen words or spelling typos that weren't in its training dictionary.

3. Pre-trained & Lookup (The Standard Usage) . Google and Stanford pre-trained models.Load a pre-trained Word2Vec file (e.g., word2vec-google-news-300), it acts as a static dictionary lookup table. Cons: If text contains industry jargon, clinical terms, or misspellings not present in the Google News vocabulary, the model fails with an Out-Of-Vocabulary (OOV) error.

4. Train a Word2Vec model from scratch on own dataset using libraries like gensim. Pass raw text corpus (e.g., 500,000 legal contracts or customer reviews) to the Word2Vec algorithm.  It scans through text using a sliding window, updates its internal weight matrix, and learns vectors tailored specifically to  domain's vocabulary. 

**GloVE** 

1. While Word2Vec streams through text looking at small local context windows, GloVe looks at the global co-occurrence statistics across the entire dataset at once. It works on the principle that the ratio of co-occurrence probabilities between words captures underlying semantic relationships:

Consider $x = \text{"ice"}$ and $y = \text{"steam"}$:For $k = \text{"solid"}$: $\frac{P(\text{solid} \mid \text{ice})}{P(\text{solid} \mid \text{steam})}$ is large (ice is solid, steam is not).For $k = \text{"gas"}$: $\frac{P(\text{gas} \mid \text{ice})}{P(\text{gas} \mid \text{steam})}$ is small (steam is gas, ice is not).For $k = \text{"water"}$: $\frac{P(\text{water} \mid \text{ice})}{P(\text{water} \mid \text{steam})}$ is close to 1 (both relate to water).


> Primary Mechanism :	Predictive (Neural Net over local windows)	vs Count-based (Global Matrix Factorization). Data Usage	Streams over local sliding windows line-by-line vs	Summarizes corpus into a global co-occurrence matrix first. Training Efficiency	Faster on massive, streaming corpora	vs Extremely efficient at leveraging overall corpus statistics

**7. Imabalanced Data**

1. SMOTE (Synthetic Minority Over-sampling Technique) -  Creates artificial minority samples along line segments connecting $k$-nearest minority neighbors. Pick a minority point $\mathbf{x}_i$. Find its $k$-nearest neighbors among minority samples; pick one randomly, $\mathbf{x}_{\text{zi}}$. Generate synthetic sample:$$\mathbf{x}_{\text{new}} = \mathbf{x}_i + \lambda (\mathbf{x}_{\text{zi}} - \mathbf{x}_i) \quad \text{where } \lambda \sim U(0, 1)$$

> Traps & Pitfalls: Overfitting / Data Leakage:  Always run SMOTE only on training folds. Noise Propagation: If minority points are noise/outliers, SMOTE creates synthetic points deep inside majority class clusters.

2. Focal Loss - Adds a modulating factor $(1 - p_t)^\gamma$ to cross-entropy loss to dynamically down-weight easy examples during training:$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$(where $p_t$ is predicted probability for ground truth class, $\alpha_t$ balances class frequency, $\gamma \ge 0$ controls focusing rate)Traps & Pitfalls: Sensitive to the hyperparameter $\gamma$. If set too high, model gradients are dominated entirely by extreme outliers

3. Precision, Recall & $F_\beta$- ScoreFormulas:$$\text{Precision} = \frac{TP}{TP + FP}, \quad \text{Recall} = \frac{TP}{TP + FN}$$ 
$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \times \text{Recall}}{(\beta^2 \cdot \text{Precision}) + \text{Recall}}$$

$\beta = 1$: $F_1$-score (harmonic mean).$\beta = 2$: Weights Recall twice as heavily as Precision (ideal for medical diagnostics/fraud detection).

4. PR-AUC vs. ROC-AUC - Formulas & Logic:ROC Curve: Plots True Positive Rate ($\frac{TP}{TP+FN}$) vs. False Positive Rate ($\frac{FP}{FP+TN}$).PR Curve: Plots Precision vs. Recall. Traps & Pitfalls (ROC-AUC Trap): On datasets with extreme imbalance (e.g., 99.9% negative), the False Positive Rate denominator ($FP + TN$) is dominated by $TN$. A massive rise in absolute false positives barely changes FPR, giving a misleadingly high ROC-AUC ($>0.95$) despite poor real-world model performance. Always use PR-AUC for severe class imbalance.